# 01 – Download NSINA slice (STGWKS-13)

**One-time setup before running:**
1. Log in at huggingface.co, open `sinhala-nlp/NSINA` and accept the access conditions.
2. Create a **read** token: huggingface.co → Settings → Access Tokens.
3. In Colab, open the 🔑 Secrets panel (left sidebar), add a secret named `HF_TOKEN` with that token, and allow notebook access.

A GPU is not needed. The file (~1.9 GB) is read **one article at a time**, so memory stays low.

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys, json
DATA_DIR = '/content/drive/MyDrive/sinhala-tiny-gpt'
REPO = '/content/sinhala-tiny-gpt'
if not os.path.exists(REPO):
    !git clone https://github.com/Amitha4469/sinhala-tiny-gpt.git {REPO}
else:
    !git -C {REPO} pull
sys.path.insert(0, REPO)

!pip install -q ijson
import ijson
print('ijson backend:', ijson.backend)  # yajl2_c is the fast one

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 5), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 4.77 KiB | 1.59 MiB/s, done.
From https://github.com/Amitha4469/sinhala-tiny-gpt
   3bba61a..bbc767a  main       -> origin/main
Updating 3bba61a..bbc767a
Fast-forward
 notebooks/01_download_data.ipynb |  26 +++++---
 requirements.txt                 |   3 +-
 sgpt/data/download.py            | 135 ++++++++++++++++++++++++++++-----------
 tests/test_download.py           |  73 +++++++++++++++++----
 4 files changed, 179 insertions(+), 58 deletions(-)
ijson backend: yajl2_c


In [2]:
from sgpt.data.download import download_nsina, iter_records, make_slice, summarize, save_jsonl

# Downloads ~1.9 GB the first time; reuses the cached file afterwards
paths = download_nsina(token=userdata.get('HF_TOKEN'))
for p in paths:
    print(p, round(os.path.getsize(p) / 1e9, 2), 'GB')

/root/.cache/huggingface/hub/datasets--sinhala-nlp--NSINA/snapshots/a986af81ad1959701aaac7ee13f756845928e672/NSINA.json 1.87 GB


In [3]:
# Stream all articles and keep a seeded random 40 MB slice (a few minutes)
rows = (r for p in paths for r in iter_records(p))
records, seen = make_slice(rows, target_mb=40, seed=1234, progress_every=50_000)

summary = summarize(records)
summary['rows_in_nsina'] = seen
print(json.dumps(summary, ensure_ascii=False, indent=2))

  read 50,000 articles, kept 16,452
  read 100,000 articles, kept 17,501
  read 150,000 articles, kept 13,433
  read 200,000 articles, kept 14,529
  read 250,000 articles, kept 13,308
  read 300,000 articles, kept 13,936
  read 350,000 articles, kept 14,628
  read 400,000 articles, kept 15,018
  read 450,000 articles, kept 13,598
{
  "articles": 9187,
  "text_mb": 40.0,
  "characters": 15919713,
  "by_source": {
    "www.lankadeepa.lk": 2572,
    "hirunews": 2368,
    "Adaderana": 1269,
    "Lankatruth": 913,
    "www.vikalpa.org": 571,
    "divaina": 504,
    "ITN news": 429,
    "https://sinhala.news.lk": 354,
    "dinamina": 149,
    "Siyatha News": 58
  },
  "by_category": {
    "latest_news": 2058,
    "Local News": 1781,
    "null": 1269,
    "පුවත්": 750,
    "N/A": 571,
    "provincial-news": 274,
    "International News": 268,
    "Sports": 188,
    "දේශීය පුවත්": 186,
    "features": 128,
    "political_current_affairs": 123,
    "pradeshiya-puvath": 112,
    "Business": 100,

In [4]:
out_path = f'{DATA_DIR}/raw/nsina_slice_40mb_seed1234.jsonl'
save_jsonl(records, out_path)
with open(f'{DATA_DIR}/raw/nsina_slice_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved', len(records), 'articles to', out_path)

Saved 9187 articles to /content/drive/MyDrive/sinhala-tiny-gpt/raw/nsina_slice_40mb_seed1234.jsonl
